# 01 · Setup, data check and cohort audit

Upload the Aspera download to `MyDrive/bmets/UCSD-BMETS-Longitudinal/` and the four TSVs to `MyDrive/bmets/clinical/`. You need about 350 GB of Drive space (raw data plus the preprocessed cache). Otherwise run preprocessing on Juno and copy `cache/pp` to Drive.

In [ ]:
# --- setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['BMF_DRIVE'] = '/content/drive/MyDrive/bmets'                       # change if you use another folder
os.environ['BMF_DATA'] = os.environ['BMF_DRIVE'] + '/UCSD-BMETS-Longitudinal'   # Aspera download target (uploaded to Drive)
os.makedirs(os.environ['BMF_DRIVE'], exist_ok=True)
REPO = '/content/claude_projects'
if not os.path.exists(REPO):
    !git clone -q https://github.com/rittik047/claude_projects.git $REPO
%cd /content/claude_projects/bmets-forecast
!pip -q install -e ".[dl,extra]"
CFG = 'configs/colab.yaml'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pytest -q -p no:warnings tests  # synthetic tests: must pass

In [ ]:
!bmf audit --config $CFG
!bmf index --config $CFG
!bmf splits --config $CFG

In [ ]:
import json; print(json.dumps(json.load(open(os.environ['BMF_DRIVE'] + '/reports/audit.json')), indent=1)[:2000])

**Check the label values** of the first segmentation before continuing. Expected: {0,1,2,3}, plus 4 for a resection cavity.

In [ ]:
import glob, nibabel as nib, numpy as np
f = sorted(glob.glob(os.environ['BMF_DATA'] + '/*/*seg*.nii.gz'))[0]
print(f, np.unique(nib.load(f).get_fdata()))